# Geometric EEG SSL v2 — Cross-Montage Pretrain Notebook (GPU)

**Purpose:** Pretrain v2 with mixed cross-montage corpus, three
leave-one-dataset-out runs. Headline experiment for v2.

**Prereq:** Run `colab_download.ipynb` first to cache all three datasets
(PhysioNet MI, BCIC-2B, Sleep-EDFx) and their preprocessed signal arrays.
v2 reuses v1's signal cache unchanged — only `ch_pos` is recomputed
on the fly via `src/v2/preprocess.ch_pos_from_names` (fixed-scale,
shared across montages).

Each pretrain cell auto-resumes from the latest checkpoint if interrupted.


## 1. Install dependencies

In [1]:
%%capture
!pip install mne moabb scikit-learn pyyaml scipy


## 2. Mount Google Drive + paths

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/geometric_eeg_ssl'
os.makedirs(DRIVE_ROOT, exist_ok=True)

# MNE data dir (raw downloads from colab_download.ipynb)
MNE_DATA_DIR = f'{DRIVE_ROOT}/mne_data'
os.environ['MNE_DATA'] = MNE_DATA_DIR

# v1 signal cache; reused unchanged by v2.
CACHE_ROOT = f'{DRIVE_ROOT}/cache'
os.environ['EEG_CACHE_DIR'] = CACHE_ROOT
if not os.path.isdir(CACHE_ROOT):
    print(f'WARNING: no cache at {CACHE_ROOT}. Loaders will reprocess; '
          'run colab_download.ipynb section 5 to build the cache once.')
else:
    print(f'Reusing v1 signal cache at {CACHE_ROOT}')

# v2 checkpoints go to a dedicated subtree so they don't shadow v1's.
CKPT_ROOT = f'{DRIVE_ROOT}/runs/pretrain'
os.makedirs(CKPT_ROOT, exist_ok=True)
print(f'Checkpoints -> {CKPT_ROOT}')


Mounted at /content/drive
Reusing v1 signal cache at /content/drive/MyDrive/geometric_eeg_ssl/cache
Checkpoints -> /content/drive/MyDrive/geometric_eeg_ssl/runs/pretrain


## 2b. Keep Colab alive

Colab sessions die when the browser tab loses focus or the laptop sleeps.
Checkpoints saved every 10 epochs keep work recoverable, but the keep-alive
ping below buys uninterrupted long runs. Re-run after any browser refresh.


In [3]:
from IPython.display import display, Javascript
display(Javascript('''
function ClickConnect() {
  const btn = document.querySelector("colab-connect-button");
  if (btn && btn.shadowRoot) {
    const inner = btn.shadowRoot.querySelector("#connect");
    if (inner) inner.click();
  }
  console.log("colab keep-alive ping " + new Date().toLocaleTimeString());
}
if (window._colabKeepAlive) clearInterval(window._colabKeepAlive);
window._colabKeepAlive = setInterval(ClickConnect, 60000);
console.log("colab keep-alive armed (60s interval)");
'''))
print('Keep-alive armed.')


<IPython.core.display.Javascript object>

Keep-alive armed.


## 3. Clone repo (v2-improvements branch)

In [4]:
import os, sys
REPO_DIR = '/content/geometric-eeg-ssl'
if not os.path.exists(REPO_DIR):
    !git clone -b v2-improvements https://github.com/tianxin-scu/geometric-eeg-ssl.git {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch && git -C {REPO_DIR} checkout v2-improvements && git -C {REPO_DIR} pull
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, f'{REPO_DIR}/src')
print('Repo ready at', REPO_DIR, '(branch v2-improvements)')


Cloning into '/content/geometric-eeg-ssl'...
remote: Enumerating objects: 270, done.
remote: Counting objects: 100% (270/270), done.
remote: Compressing objects: 100% (184/184), done.
remote: Total 270 (delta 127), reused 207 (delta 73), pack-reused 0 (from 0)
Receiving objects: 100% (270/270), 896.01 KiB | 14.45 MiB/s, done.
Resolving deltas: 100% (127/127), done.
Repo ready at /content/geometric-eeg-ssl (branch v2-improvements)


## 4. Verify the v2 stack imports + smoke

Quick sanity check the v2 modules import cleanly. If this fails, every
pretrain cell below will too -- fix imports first.


In [5]:
import subprocess
out = subprocess.run(
    ['python', f'{REPO_DIR}/tests/smoke_test_v2.py'],
    cwd=REPO_DIR, capture_output=True, text=True,
    env={**os.environ, 'PYTHONPATH': REPO_DIR},
)
print(out.stdout)
if out.returncode != 0:
    print('STDERR:', out.stderr)
    raise RuntimeError('v2 smoke test failed')


smoke OK; 4 mixed-corpus steps, losses[-1]=('bcic_2b', 0.7913130521774292, -0.28106310963630676, 1.0723761320114136)
zero-shot to held-out sleep_edfx (M=2): backbone output shape (2, 2, 16, 256)



## 4b. Stage caches to local disk

Google Drive FUSE mounts drop under sustained read pressure -- reading
multi-GB compressed `.npz` files from Drive frequently throws
`ConnectionAbortedError: [Errno 103]`. The fix is to copy caches to local
Colab disk once at session start, point `EEG_CACHE_DIR` at the local copy,
and let training read locally.

This cell also builds the Sleep-EDFx cache if missing (reads raw EDFs from
Drive once, writes cache locally, copies it back to Drive for next session).
First-run cost: ~15 min for the Sleep-EDFx build.

Re-run after any session restart -- the local disk is wiped between sessions.


In [ ]:
import os, glob, time, sys, shutil

LOCAL_CACHE = '/content/cache_local'
os.makedirs(LOCAL_CACHE, exist_ok=True)

# Robust chunked copy with retry + Drive remount on transient FUSE drops.
# shutil.copy uses os.sendfile which fails hard on errno 107; chunked reads
# in user space recover better, and re-mounting Drive heals after a drop.
def _remount_drive():
    from google.colab import drive
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    drive.mount('/content/drive', force_remount=True)
    time.sleep(2)

def robust_copy(src, dst, chunk_mb=4, max_retries=6):
    chunk = chunk_mb * 1024 * 1024
    for attempt in range(max_retries):
        try:
            # If a partial file is on disk from a prior attempt, drop it.
            if os.path.exists(dst):
                os.remove(dst)
            with open(src, 'rb') as fi, open(dst, 'wb') as fo:
                while True:
                    buf = fi.read(chunk)
                    if not buf:
                        break
                    fo.write(buf)
            return True
        except OSError as e:
            if e.errno in (103, 107) and attempt < max_retries - 1:
                wait = 5 * (2 ** attempt)  # 5, 10, 20, 40, 80 s
                print(f'    [errno {e.errno}] retry {attempt+1}/{max_retries-1} '
                      f'after {wait}s + drive remount...')
                _remount_drive()
                time.sleep(wait)
                continue
            raise
    return False

# Stage existing Drive caches to local disk (pretrain only -- eval caches
# are not used by v2 pretrain).
WANTED_PREFIXES = ('physionet_mi_pretrain_', 'bcic_2b_pretrain_', 'sleep_edfx_pretrain_')
srcs = [p for p in sorted(glob.glob(f'{CACHE_ROOT}/*.npz'))
        if any(os.path.basename(p).startswith(w) for w in WANTED_PREFIXES)]
print(f'staging {len(srcs)} pretrain cache file(s) to {LOCAL_CACHE}')
for src in srcs:
    dst = os.path.join(LOCAL_CACHE, os.path.basename(src))
    if os.path.exists(dst):
        try:
            same = os.path.getsize(dst) == os.path.getsize(src)
        except OSError:
            same = False
        if same:
            print(f'  already staged: {os.path.basename(src)}')
            continue
    t0 = time.time()
    robust_copy(src, dst)
    print(f'  staged {os.path.basename(src)}  '
          f'({os.path.getsize(dst)/1e6:.1f} MB, {time.time()-t0:.1f}s)')

# Repoint EEG_CACHE_DIR -- the v2 training script reads this env var.
os.environ['EEG_CACHE_DIR'] = LOCAL_CACHE
print(f'\nEEG_CACHE_DIR -> {LOCAL_CACHE}')

# Build Sleep-EDFx cache locally if missing. Reads raw EDFs from Drive once.
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, f'{REPO_DIR}/src')
sleep_cached = any('sleep_edfx_pretrain_' in f for f in os.listdir(LOCAL_CACHE))
if not sleep_cached:
    print('\nSleep-EDFx cache missing -- building locally (~15 min)...')
    from src.config import Config
    from src.datasets.sleep_edfx import ALL_SUBJECTS, SleepEDFx
    cfg = Config()
    # Cap to N subjects to keep RAM under the Colab limit (~12 GB).
    # The v2 sampler caps each dataset at EPOCHS_PER_DATASET per pass, so
    # ~600k sub-epochs from 30 subjects is more than enough.
    SLEEP_N_SUBJECTS = 30
    subj_subset = list(ALL_SUBJECTS)[:SLEEP_N_SUBJECTS]
    print(f'using {len(subj_subset)} of {len(ALL_SUBJECTS)} sleep subjects '
          f'(memory-bounded; v2 sampler subsamples anyway)')
    SleepEDFx(subjects=subj_subset, cfg=cfg,
              mode='pretrain', verbose=True).load()
    # Back up to Drive (robust copy; surviving the round trip is bonus).
    for src in glob.glob(f'{LOCAL_CACHE}/sleep_edfx_pretrain_*.npz'):
        dst = os.path.join(CACHE_ROOT, os.path.basename(src))
        if not os.path.exists(dst):
            try:
                robust_copy(src, dst)
                print(f'  backed up {os.path.basename(src)} to Drive')
            except OSError as e:
                print(f'  warning: Drive backup failed (errno {e.errno}); '
                      f'cache still available locally for this session')
else:
    print('Sleep-EDFx cache already present locally.')

print('\nLocal cache ready.')


## 5. Verify all three datasets are cached

v2 needs all three for the three leave-one-out splits. If any is missing,
run the corresponding section of `colab_download.ipynb`.


In [6]:
import os, glob

# v2 reuses v1's preprocessing cache. The pretrain script never needs the
# raw MNE downloads if the cache fingerprints match -- so we accept either.
raw_paths = {
    'PhysioNet MI': os.path.join(MNE_DATA_DIR, 'MNE-eegbci-data', 'files', 'eegmmidb', '1.0.0'),
    'BCIC-2B':      os.path.join(MNE_DATA_DIR, 'MNE-bnci-data'),
    'Sleep-EDFx':   os.path.join(MNE_DATA_DIR, 'physionet-sleep-data'),
}
cache_prefixes = {
    'PhysioNet MI': 'physionet_mi_pretrain_',
    'BCIC-2B':      'bcic_2b_pretrain_',
    'Sleep-EDFx':   'sleep_edfx_pretrain_',
}

cache_files = set(os.listdir(CACHE_ROOT)) if os.path.isdir(CACHE_ROOT) else set()

missing = []
for name in raw_paths:
    has_raw = os.path.isdir(raw_paths[name])
    has_cache = any(f.startswith(cache_prefixes[name]) for f in cache_files)
    status = []
    if has_cache:
        status.append('cache')
    if has_raw:
        status.append('raw')
    if not status:
        status = ['MISSING']
        missing.append(name)
    print(f'  {name:14s}  [{" + ".join(status)}]')

if missing:
    raise RuntimeError(
        f'Missing both cache and raw for: {missing}. '
        'Run colab_download.ipynb sections 4 (download) and 5 (build cache).'
    )
print('All three datasets have either a cache or raw download present.')


  PhysioNet MI    [cache + raw]
  BCIC-2B         [cache + raw]
  Sleep-EDFx      [raw]
All three datasets have either a cache or raw download present.


## 6. Per-dataset budget

The three datasets have very different epoch counts:

| Dataset | Epochs available | Channels |
|---|---|---|
| PhysioNet MI | ~9,450 | 64 |
| BCIC-2B | ~6,500 | 3 |
| Sleep-EDFx | ~1,000,000 | 2 (bipolar) |

The v2.2 claim is "distribution of g_ij matters, count doesn't" -- so we cap
each dataset at the same budget per pass. Small datasets are oversampled
with replacement; Sleep-EDFx is subsampled hard.

`EPOCHS_PER_DATASET = 8000` gives roughly the same wall-clock per pass as v1
single-dataset pretraining (which ran ~9,450 PhysioNet epochs/epoch). Bump
or lower as compute allows.


In [7]:
EPOCHS_PER_DATASET = 8000
print(f'budget per dataset per pass = {EPOCHS_PER_DATASET}')


budget per dataset per pass = 8000


## 7. Pretrain v2 -- `no_sleep` (Sleep-EDFx held out)

Mixed corpus: `physionet_mi,bcic_2b`. Held out: the third dataset, for downstream
zero-shot eval. Checkpoints saved every 10 epochs to Drive.


In [8]:
import os, glob

CKPT_DIR = f'{CKPT_ROOT}/v2_no_sleep'
os.makedirs(CKPT_DIR, exist_ok=True)

_mode = 'resuming' if glob.glob(f'{CKPT_DIR}/epoch_*.pt') else 'starting fresh'
print(f'v2_no_sleep: {_mode}')

!python -u {REPO_DIR}/scripts/v2_pretrain.py \
    --config {REPO_DIR}/configs/v2/pretrain/v2_default.yaml \
    --pretrain-datasets physionet_mi,bcic_2b \
    --epochs-per-dataset {EPOCHS_PER_DATASET} \
    --ckpt-dir {CKPT_DIR} \
    --device cuda \
    --resume latest \
    2>&1 | tee -a /content/train_log_$(basename {CKPT_DIR}).txt


v2_no_sleep: starting fresh
pretrain on: ['physionet_mi', 'bcic_2b']
held out (for downstream zero-shot eval): ['sleep_edfx']
device: cuda

--- loading physionet_mi ---
[cache] loading physionet_mi/pretrain from physionet_mi_pretrain_1bfb941d013e1e1e.npz
  X=(9408, 64, 800)  M=64  ch_names[0..5]=['FC5', 'FC3', 'FC1', 'FCz', 'FC2']

--- loading bcic_2b ---
[cache] loading bcic_2b/pretrain from bcic_2b_pretrain_9c577d3abbf5db8d.npz
Traceback (most recent call last):
  File "/content/geometric-eeg-ssl/scripts/v2_pretrain.py", line 351, in <module>
tee: /content/drive/MyDrive/geometric_eeg_ssl/runs/pretrain/v2_no_sleep/train_log.txt: Transport endpoint is not connected
    main()
  File "/content/geometric-eeg-ssl/scripts/v2_pretrain.py", line 238, in main
    X, pos, ch_names = _load_dataset(name, cfg)
                       ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/geometric-eeg-ssl/scripts/v2_pretrain.py", line 105, in _load_dataset
    ).load()
      ^^^^^^
  File "/content/geometric-e

## 7. Pretrain v2 -- `no_bcic` (BCIC-2B held out)

Mixed corpus: `physionet_mi,sleep_edfx`. Held out: the third dataset, for downstream
zero-shot eval. Checkpoints saved every 10 epochs to Drive.


In [ ]:
import os, glob

CKPT_DIR = f'{CKPT_ROOT}/v2_no_bcic'
os.makedirs(CKPT_DIR, exist_ok=True)

_mode = 'resuming' if glob.glob(f'{CKPT_DIR}/epoch_*.pt') else 'starting fresh'
print(f'v2_no_bcic: {_mode}')

!python -u {REPO_DIR}/scripts/v2_pretrain.py \
    --config {REPO_DIR}/configs/v2/pretrain/v2_default.yaml \
    --pretrain-datasets physionet_mi,sleep_edfx \
    --epochs-per-dataset {EPOCHS_PER_DATASET} \
    --ckpt-dir {CKPT_DIR} \
    --device cuda \
    --resume latest \
    2>&1 | tee -a /content/train_log_$(basename {CKPT_DIR}).txt


v2_no_bcic: starting fresh
pretrain on: ['physionet_mi', 'sleep_edfx']
held out (for downstream zero-shot eval): ['bcic_2b']
device: cuda

--- loading physionet_mi ---
[cache] loading physionet_mi/pretrain from physionet_mi_pretrain_1bfb941d013e1e1e.npz
  X=(9408, 64, 800)  M=64  ch_names[0..5]=['FC5', 'FC3', 'FC1', 'FCz', 'FC2']

--- loading sleep_edfx ---
[cache] building sleep_edfx/pretrain; will save to sleep_edfx_pretrain_e559c3702ac00e92.npz
/content/geometric-eeg-ssl/src/datasets/sleep_edfx.py:209: RuntimeWarning: Requested recording 1 for subject 36 and/or 52, but it is not available in corpus.
  files = mne.datasets.sleep_physionet.age.fetch_data(
/content/geometric-eeg-ssl/src/datasets/sleep_edfx.py:209: RuntimeWarning: Requested recording 2 for subject 13, but it is not available in corpus.
  files = mne.datasets.sleep_physionet.age.fetch_data(
/content/geometric-eeg-ssl/src/datasets/sleep_edfx.py:144: RuntimeWarning: Channels contain different highpass filters. Highest filt

## 7. Pretrain v2 -- `no_phys` (PhysioNet MI held out)

Mixed corpus: `bcic_2b,sleep_edfx`. Held out: the third dataset, for downstream
zero-shot eval. Checkpoints saved every 10 epochs to Drive.


In [ ]:
import os, glob

CKPT_DIR = f'{CKPT_ROOT}/v2_no_phys'
os.makedirs(CKPT_DIR, exist_ok=True)

_mode = 'resuming' if glob.glob(f'{CKPT_DIR}/epoch_*.pt') else 'starting fresh'
print(f'v2_no_phys: {_mode}')

!python -u {REPO_DIR}/scripts/v2_pretrain.py \
    --config {REPO_DIR}/configs/v2/pretrain/v2_default.yaml \
    --pretrain-datasets bcic_2b,sleep_edfx \
    --epochs-per-dataset {EPOCHS_PER_DATASET} \
    --ckpt-dir {CKPT_DIR} \
    --device cuda \
    --resume latest \
    2>&1 | tee -a /content/train_log_$(basename {CKPT_DIR}).txt


## 8. Checkpoint inventory

In [ ]:
import glob, os

for tag in ['no_sleep', 'no_bcic', 'no_phys']:
    d = f'{CKPT_ROOT}/v2_{tag}'
    ckpts = sorted(glob.glob(f'{d}/epoch_*.pt'))
    last = ckpts[-1].split('/')[-1] if ckpts else 'NONE'
    spec = os.path.join(d, 'v2_run.txt')
    spec_str = open(spec).read().strip() if os.path.exists(spec) else '(no v2_run.txt)'
    print(f'v2_{tag:9s}  ckpts={len(ckpts):3d}  latest={last}')
    print('  ' + spec_str.replace('\n', '\n  '))
    print()


## Done

Three v2 pretrain runs complete. Each `runs/pretrain/v2_<tag>/v2_run.txt`
records which dataset is held out for downstream zero-shot eval.

Next: switch to the v2 experiment notebook (to be written) for probe
evaluation on the held-out dataset per run.
